# Thử nghiệm sơ bộ: phân loại SMS tiếng Việt 3 nhãn

Đề tài: Phân loại tin nhắn SMS lừa đảo và quảng cáo tiếng Việt bằng PhoBERT (nhóm 5, PPNCKH).

Notebook này so sánh ba mô hình trên 600 tin đợt 1 đã được người duyệt chốt nhãn:

| Mô hình | Vai trò | Vì sao |
| --- | --- | --- |
| TF-IDF + Logistic Regression | Mô hình cơ sở | Đơn giản, chạy vài giây, là mốc để so |
| PhoBERT-base | Mô hình chính | Mô hình ngôn ngữ tiếng Việt dùng nhiều nhất |
| ViSoBERT | Mô hình so sánh | Học trên văn bản mạng xã hội, được báo là chịu văn bản không dấu tốt hơn |

Câu hỏi nghiên cứu notebook trả lời được (ở mức sơ bộ): CH2 (so F1 lớp lừa đảo), CH3 (nhầm lẫn lừa đảo và quảng cáo), CH4 (F1 giảm bao nhiêu khi bỏ dấu).

## Cách chạy

1. Mở notebook trên Google Colab. Vào **Runtime → Change runtime type → T4 GPU**.
2. Chọn **Runtime → Run all**.
3. Khi ô "Tải dữ liệu" hiện nút **Choose Files**, chọn file `gan-nhan-so-bo.csv` đã điền cột `nhan_chot`.
4. Chờ khoảng 15 đến 25 phút. Ô cuối tự tải file `ket-qua-so-bo.json` về máy. Chép file này vào `nghien-cuu/ket-qua/` trong repo.

Kết quả chỉ là thử nghiệm sơ bộ trên 600 tin: đủ để cho thấy quy trình chạy được, chưa đủ để kết luận mô hình nào tốt hơn.

## 1. Cài thư viện

Colab đã có sẵn `torch`, `transformers`, `scikit-learn`. Cài thêm công cụ tách từ cho PhoBERT.

In [ ]:
!pip -q install py_vncorenlp underthesea
!java -version

## 2. Cấu hình

Mọi lựa chọn nằm ở ô này, để ghi vào mục 12 của thuyết minh và để người khác chạy lại ra cùng kết quả.
`SEED` cố định cách chia dữ liệu và khởi tạo mô hình.

In [ ]:
SEED = 42
TEN_FILE = "gan-nhan-so-bo.csv"
COT_NHAN = "nhan_chot"          # cột nhãn người duyệt đã chốt
CHI_DOT_1 = True                # chỉ dùng 600 tin đợt 1
NHAN = ["LUA_DAO", "QUANG_CAO", "BINH_THUONG"]
SO_FOLD = 5

# tên mô hình trên Hugging Face, có cần tách từ trước không
MO_HINH = {
    "PhoBERT-base": ("vinai/phobert-base", True),
    "ViSoBERT": ("uitnlp/visobert", False),
}
EPOCHS = 4
BATCH = 16
LR = 2e-5
MAX_LEN = 128                   # tin SMS ngắn; PhoBERT cho tối đa 256

## 3. Tải dữ liệu

In [ ]:
import os
if not os.path.exists(TEN_FILE):
    from google.colab import files
    files.upload()

## 4. Đọc và lọc dữ liệu

Bỏ các tin chưa có nhãn chốt hoặc gán `KHONG_RO`. Đếm số tin mỗi nhãn và tỉ lệ tin không dấu.

In [ ]:
import random, re, unicodedata, json, datetime
import numpy as np
import pandas as pd

df = pd.read_csv(TEN_FILE, encoding="utf-8-sig", dtype=str).fillna("")
if CHI_DOT_1:
    df = df[df["dot"] == "1"]
df[COT_NHAN] = df[COT_NHAN].str.strip().str.upper()
bo_qua = df[~df[COT_NHAN].isin(NHAN)]
df = df[df[COT_NHAN].isin(NHAN)].reset_index(drop=True)
print(f"Dùng {len(df)} tin. Bỏ qua {len(bo_qua)} tin (trống hoặc KHONG_RO).")


def bo_dau(s):
    """Bỏ dấu thanh và dấu mũ: 'Ngân hàng' -> 'Ngan hang'."""
    s = s.replace("đ", "d").replace("Đ", "D")
    s = unicodedata.normalize("NFD", s)
    return "".join(c for c in s if unicodedata.category(c) != "Mn")


df["khong_dau"] = df["message"] == df["message"].map(bo_dau)
thong_ke = pd.DataFrame({
    "so_tin": df[COT_NHAN].value_counts(),
    "ti_le_khong_dau": df.groupby(COT_NHAN)["khong_dau"].mean().round(3),
}).reindex(NHAN)
print(thong_ke)

print("\nNhãn gốc 2 lớp (0 = hợp lệ, 1 = rác/lừa đảo) so với nhãn 3 lớp đã chốt:")
bang_goc = pd.crosstab(df["nhan_goc"], df[COT_NHAN]).reindex(columns=NHAN, fill_value=0)
print(bang_goc)

dung_goi_y = (df["nhan_goi_y"] == df[COT_NHAN]).mean()
print(f"\nTỉ lệ nhãn gợi ý của script trùng nhãn chốt: {dung_goi_y:.1%}")

## 5. Chia 5 phần (5-fold, phân tầng theo nhãn)

Mỗi lần lấy 1 phần làm tập kiểm tra, 4 phần còn lại để huấn luyện, lặp 5 lần rồi lấy trung bình.
Phân tầng nghĩa là phần nào cũng có tỉ lệ ba nhãn giống nhau.

In [ ]:
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import precision_recall_fscore_support, f1_score, confusion_matrix

X = df["message"].tolist()
X_bo_dau = [bo_dau(s) for s in X]
y = df[COT_NHAN].tolist()
folds = list(StratifiedKFold(n_splits=SO_FOLD, shuffle=True, random_state=SEED).split(X, y))

ket_qua = []      # mỗi dòng: một mô hình, một fold, một kiểu tập kiểm tra
ma_tran = {}      # cộng dồn ma trận nhầm lẫn qua 5 fold


def ghi(mo_hinh, fold, kieu, y_that, y_doan):
    p, r, f, _ = precision_recall_fscore_support(y_that, y_doan, labels=NHAN, zero_division=0)
    dong = {"mo_hinh": mo_hinh, "fold": fold, "kieu": kieu,
            "macro_f1": f1_score(y_that, y_doan, labels=NHAN, average="macro", zero_division=0)}
    for i, n in enumerate(NHAN):
        dong[f"P_{n}"], dong[f"R_{n}"], dong[f"F1_{n}"] = p[i], r[i], f[i]
    ket_qua.append(dong)
    cm = confusion_matrix(y_that, y_doan, labels=NHAN)
    ma_tran[(mo_hinh, kieu)] = ma_tran.get((mo_hinh, kieu), 0) + cm

## 6. Mô hình cơ sở: TF-IDF + Logistic Regression

TF-IDF biến mỗi tin thành một vectơ: từ nào xuất hiện nhiều trong tin này nhưng hiếm trong các tin khác thì có trọng số cao.
Logistic Regression học trọng số cho từng từ để đoán nhãn. Ví dụ từ "xác minh" và "bit.ly" có thể kéo tin về phía `LUA_DAO`.
`kieu = "bo_dau"` là cùng tập kiểm tra nhưng đã bỏ dấu, để trả lời CH4.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline

for k, (tr, te) in enumerate(folds):
    lr = make_pipeline(
        TfidfVectorizer(ngram_range=(1, 2), sublinear_tf=True),
        LogisticRegression(max_iter=2000, class_weight="balanced"),
    )
    lr.fit([X[i] for i in tr], [y[i] for i in tr])
    y_te = [y[i] for i in te]
    ghi("TF-IDF + LR", k, "goc", y_te, lr.predict([X[i] for i in te]))
    ghi("TF-IDF + LR", k, "bo_dau", y_te, lr.predict([X_bo_dau[i] for i in te]))
print("Xong mô hình cơ sở.")

## 7. Tách từ cho PhoBERT

PhoBERT học trên văn bản đã tách từ, ví dụ "ngân hàng" thành "ngân_hàng". Bài gốc dùng RDRSegmenter của VnCoreNLP (cần Java).
Nếu VnCoreNLP không chạy được, notebook dùng `underthesea` và in ra thông báo. Ghi lại công cụ đã dùng vào mục 12.

In [ ]:
def tao_tach_tu():
    try:
        import py_vncorenlp
        thu_muc = "/content/vncorenlp"
        os.makedirs(thu_muc, exist_ok=True)
        if not os.path.exists(os.path.join(thu_muc, "models")):
            py_vncorenlp.download_model(save_dir=thu_muc)
        cwd = os.getcwd()
        rdr = py_vncorenlp.VnCoreNLP(annotators=["wseg"], save_dir=thu_muc)
        os.chdir(cwd)  # VnCoreNLP tự đổi thư mục làm việc, đổi lại
        return "RDRSegmenter (VnCoreNLP)", lambda s: " ".join(rdr.word_segment(s))
    except Exception as e:
        print("Không chạy được VnCoreNLP:", e)
        from underthesea import word_tokenize
        return "underthesea", lambda s: word_tokenize(s, format="text")


CONG_CU_TACH_TU, tach_tu = tao_tach_tu()
print("Công cụ tách từ:", CONG_CU_TACH_TU)
X_tach = [tach_tu(s) for s in X]
X_tach_bo_dau = [tach_tu(s) for s in X_bo_dau]
print("Ví dụ:", X_tach[0][:120])

## 8. PhoBERT-base và ViSoBERT

Fine-tune nghĩa là lấy mô hình đã học tiếng Việt từ trước, rồi dạy thêm cho nó việc đoán 3 nhãn trên dữ liệu của mình.
Mỗi fold huấn luyện lại từ đầu. Mỗi mô hình mất khoảng 5 đến 10 phút trên GPU T4.

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup

thiet_bi = "cuda" if torch.cuda.is_available() else "cpu"
print("Thiết bị:", thiet_bi)
if thiet_bi == "cpu":
    print("CẢNH BÁO: chưa bật GPU, sẽ rất chậm. Vào Runtime -> Change runtime type -> T4 GPU.")
so_cua_nhan = {n: i for i, n in enumerate(NHAN)}


def dat_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)


def huan_luyen(ten_hf, X_tr, y_tr):
    tok = AutoTokenizer.from_pretrained(ten_hf)
    model = AutoModelForSequenceClassification.from_pretrained(ten_hf, num_labels=len(NHAN)).to(thiet_bi)
    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)
    so_buoc = EPOCHS * ((len(X_tr) + BATCH - 1) // BATCH)
    lich = get_linear_schedule_with_warmup(opt, int(0.1 * so_buoc), so_buoc)
    thu_tu = list(range(len(X_tr)))
    model.train()
    for _ in range(EPOCHS):
        random.shuffle(thu_tu)
        for i in range(0, len(thu_tu), BATCH):
            lo = thu_tu[i:i + BATCH]
            enc = tok([X_tr[j] for j in lo], truncation=True, max_length=MAX_LEN,
                      padding=True, return_tensors="pt").to(thiet_bi)
            nhan = torch.tensor([so_cua_nhan[y_tr[j]] for j in lo], device=thiet_bi)
            loss = model(**enc, labels=nhan).loss
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step(); lich.step(); opt.zero_grad()
    return model, tok


def du_doan(model, tok, texts):
    model.eval()
    ra = []
    with torch.no_grad():
        for i in range(0, len(texts), 64):
            enc = tok(texts[i:i + 64], truncation=True, max_length=MAX_LEN,
                      padding=True, return_tensors="pt").to(thiet_bi)
            ra += model(**enc).logits.argmax(-1).tolist()
    return [NHAN[i] for i in ra]


for ten, (ten_hf, can_tach) in MO_HINH.items():
    X_g = X_tach if can_tach else X
    X_b = X_tach_bo_dau if can_tach else X_bo_dau
    for k, (tr, te) in enumerate(folds):
        dat_seed(SEED + k)
        model, tok = huan_luyen(ten_hf, [X_g[i] for i in tr], [y[i] for i in tr])
        y_te = [y[i] for i in te]
        ghi(ten, k, "goc", y_te, du_doan(model, tok, [X_g[i] for i in te]))
        ghi(ten, k, "bo_dau", y_te, du_doan(model, tok, [X_b[i] for i in te]))
        print(f"{ten} fold {k + 1}/{SO_FOLD}: macro-F1 = {ket_qua[-2]['macro_f1']:.3f}")
        del model
        torch.cuda.empty_cache()

## 9. Kết quả

F1 là trung bình điều hòa của precision và recall. Với lớp lừa đảo:
precision cao nghĩa là mô hình báo "lừa đảo" thì ít khi báo nhầm; recall cao nghĩa là ít bỏ sót tin lừa đảo.
Số ghi dạng trung bình ± độ lệch chuẩn qua 5 fold.

In [ ]:
kq = pd.DataFrame(ket_qua)
cot = ["macro_f1", "P_LUA_DAO", "R_LUA_DAO", "F1_LUA_DAO"]
tb, dl = kq.groupby(["mo_hinh", "kieu"])[cot].mean(), kq.groupby(["mo_hinh", "kieu"])[cot].std()
bang = (tb * 100).round(1).astype(str) + " ± " + (dl * 100).round(1).astype(str)
print("CH2. Kết quả trên tập kiểm tra gốc (%):")
print(bang.xs("goc", level="kieu"))

print("\nCH3. Nhầm lẫn giữa lừa đảo và quảng cáo (cộng 5 fold, tập gốc):")
iL, iQ = NHAN.index("LUA_DAO"), NHAN.index("QUANG_CAO")
nham = {}
for (ten, kieu), cm in ma_tran.items():
    if kieu != "goc":
        continue
    nham[ten] = {
        "lua_dao_thanh_quang_cao_%": round(100 * cm[iL, iQ] / cm[iL].sum(), 1),
        "quang_cao_thanh_lua_dao_%": round(100 * cm[iQ, iL] / cm[iQ].sum(), 1),
    }
    print(f"\n{ten}")
    print(pd.DataFrame(cm, index=[f"thật {n}" for n in NHAN], columns=[f"đoán {n}" for n in NHAN]))
print(pd.DataFrame(nham).T)

print("\nCH4. F1 lớp lừa đảo khi bỏ dấu tập kiểm tra (điểm %):")
f1 = tb["F1_LUA_DAO"].unstack("kieu") * 100
f1["giam"] = f1["goc"] - f1["bo_dau"]
print(f1.round(1))

## 10. Lưu kết quả

File JSON ghi đủ cấu hình, phiên bản thư viện và số liệu từng fold, để đưa vào thuyết minh và để chạy lại kiểm tra.

In [ ]:
import sklearn, transformers
ra = {
    "ngay_chay": datetime.datetime.now().isoformat(timespec="seconds"),
    "cau_hinh": {"SEED": SEED, "SO_FOLD": SO_FOLD, "EPOCHS": EPOCHS, "BATCH": BATCH, "LR": LR,
                 "MAX_LEN": MAX_LEN, "MO_HINH": MO_HINH, "CHI_DOT_1": CHI_DOT_1,
                 "cong_cu_tach_tu": CONG_CU_TACH_TU, "thiet_bi": thiet_bi},
    "phien_ban": {"torch": torch.__version__, "transformers": transformers.__version__,
                  "scikit-learn": sklearn.__version__, "pandas": pd.__version__},
    "du_lieu": {"so_tin": len(df), "bo_qua": len(bo_qua),
                "thong_ke": thong_ke.reset_index().to_dict("records"),
                "nhan_goc_x_nhan_chot": bang_goc.to_dict(),
                "ti_le_goi_y_dung": round(float(dung_goi_y), 4)},
    "ket_qua_tung_fold": ket_qua,
    "ma_tran_nham_lan": {f"{t}|{k}": cm.tolist() for (t, k), cm in ma_tran.items()},
    "nham_lua_dao_quang_cao": nham,
}
with open("ket-qua-so-bo.json", "w", encoding="utf-8") as f:
    json.dump(ra, f, ensure_ascii=False, indent=2, default=float)
try:
    from google.colab import files
    files.download("ket-qua-so-bo.json")
except ImportError:
    print("Đã ghi ket-qua-so-bo.json")

## 11. Cohen's kappa (chạy khi người thứ hai đã gán xong)

Tải lên file `mau-kappa-100.csv` mà người thứ hai đã điền cột `nhan`. Ô này so nhãn của họ với cột `nhan_chot` của Trúc.
Kappa bằng 1 là hai người hoàn toàn đồng ý; bằng 0 là mức đồng ý chỉ ngang đoán ngẫu nhiên.

In [ ]:
from sklearn.metrics import cohen_kappa_score
TEN_KAPPA = "mau-kappa-100.csv"
if not os.path.exists(TEN_KAPPA):
    from google.colab import files
    files.upload()
k2 = pd.read_csv(TEN_KAPPA, encoding="utf-8-sig", dtype=str).fillna("")
goc = pd.read_csv(TEN_FILE, encoding="utf-8-sig", dtype=str).fillna("")[["id", COT_NHAN]]
gop = k2.merge(goc, on="id")
gop["nhan"] = gop["nhan"].str.strip().str.upper()
gop[COT_NHAN] = gop[COT_NHAN].str.strip().str.upper()
gop = gop[gop["nhan"].isin(NHAN) & gop[COT_NHAN].isin(NHAN)]
kappa = cohen_kappa_score(gop[COT_NHAN], gop["nhan"], labels=NHAN)
print(f"Số tin so được: {len(gop)}. Cohen's kappa = {kappa:.3f}")
print(pd.crosstab(gop[COT_NHAN], gop["nhan"]))